### Imports

In [18]:
from llama_index.core import Document, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.prompts import PromptTemplate
from dotenv import load_dotenv
import pandas as pd
import glob, os
from datetime import datetime

load_dotenv()

True

### Embedding Logic

In [ ]:
# Load all CSVs
all_files = glob.glob("data/*.csv")
print(all_files)
print(f"Found {len(all_files)} files.")

docs = []

# Set the embedding model globally
embed_model = HuggingFaceEmbedding(model_name="abhinand/MedEmbed-base-v0.1")

for f in all_files:
    df = pd.read_csv(f)
    file_name = os.path.basename(f)
    for _, row in df.iterrows():
        q, a = row["Question"], row["Answer"]
        text = f"Q: {q}\nA: {a}"
        docs.append(
            Document(
                text=text,
                metadata={
                    "file_name": file_name,
                    "created_at": datetime.now().isoformat()
                }
            )
        )

# Where to store index
persist_dir = "./vector_store"

# Build + persist
index = VectorStoreIndex.from_documents(docs, embed_model=embed_model, show_progress=True)
index.storage_context.persist(persist_dir)

print(f"✅ Index saved to {persist_dir}")

['data/Heart_Lung_and_BloodQA.csv', 'data/Diabetes_and_Digestive_and_Kidney_DiseasesQA.csv', 'data/Neurological_Disorders_and_StrokeQA.csv', 'data/Genetic_and_Rare_DiseasesQA.csv', 'data/CancerQA.csv', 'data/Disease_Control_and_PreventionQA.csv']
Found 6 files.


Parsing nodes:   0%|          | 0/9226 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1717 [00:00<?, ?it/s]

✅ Index saved to ./vector_store


#### Loading from the local Doc store (IF vector_store is generated locally)

In [14]:
storage_context = StorageContext.from_defaults(persist_dir="vector_store")

index = load_index_from_storage(
    storage_context,
    embed_model=embed_model  
)

Loading llama_index.core.storage.kvstore.simple_kvstore from vector_store/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from vector_store/index_store.json.


### Query Engine (LLM) setup

In [15]:
llm = OpenRouter(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="meta-llama/llama-3.2-3b-instruct:free",
)

qa_template = PromptTemplate(
    """
    You are a helpful medical assistant.
    Use ONLY the context below to answer the question. 
    Do not include your internal reasoning or <think> steps. 
    Just give a clear, direct answer.

    Context:
    {context}

    User Question: {input}
    Answer:
    """
)

In [16]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=5,
    max_tokens=3056,
    response_mode="compact",
    prompt=qa_template,
    streaming=True
)

#### Running the LLM with the prompt

In [21]:
def estimate_context_need(query: str) -> str:
    q = query.lower()

    # Keywords/phrases that usually mean the user expects a deep explanation
    high_context_words = [
        "why", "how", "explain", "describe", "relationship",
        "difference", "compare", "contrast", "impact", "effect",
        "causes", "reason", "pros and cons", "advantages", "disadvantages",
        "risk", "benefits", "harms", "long-term", "mechanism", "process",
        "interaction", "connection", "link", "role", "function"
    ]

    # Quick fact/short answer triggers
    low_context_words = [
        "define", "definition", "meaning", "what is", "list",
        "symptoms", "treatment", "cure", "age", "date", "year",
        "who", "where", "when"
    ]

    if any(word in q for word in high_context_words):
        return "high"
    elif any(word in q for word in low_context_words) or len(q.split()) <= 3:
        return "low"
    else:
        return "medium"


In [22]:
import json

while True:
    query = input("\nAsk a question (or 'exit'): ")
    if query.lower() == "exit":
        break

    retrieved_docs = query_engine.retriever.retrieve(query)

    # Collect only unique file names from metadata
    files_used = list({doc.node.metadata.get("file_name", "unknown") for doc in retrieved_docs})

    # Build context for the LLM
    context_texts = "\n\n".join([doc.node.text for doc in retrieved_docs])
    final_prompt = qa_template.format(context=context_texts, input=query)

    response = llm.complete(final_prompt)

    # Prepare final output
    output = {
        "question": query,
        "answer": response.text if hasattr(response, "text") else str(response),
        "files_used": files_used
    }

    print(json.dumps(output, indent=2))

{
  "question": "define cancer ",
  "answer": "Cancer is a disease in which malignant (cancer) cells form in the tissues of the body.",
  "files_used": [
    "CancerQA.csv"
  ]
}


### Check for the whole process retrieval + LLM generation 

In [ ]:
while True:
    query = input("\nAsk a question (or 'exit'): ")
    if query.lower() == "exit":
        break

    retrieved_docs = query_engine.retriever.retrieve(query)

    for i, doc in enumerate(retrieved_docs, 1):
        file_name = doc.node.metadata.get("file_name", "unknown")
        created_at = doc.node.metadata.get("created_at", "unknown")

        print(f"\n--- Retrieved {i} ---")
        print(f"File: {file_name}")
        print(f"Created: {created_at}")
        print(f"Content: {doc.node.text[:400]}")

    context_texts = "\n\n".join([doc.node.text for doc in retrieved_docs])

    final_prompt = qa_template.format(context=context_texts, input=query)

    response = llm.complete(final_prompt)

    print(f"\nQuery: {query}")
    print(f"\nAnswer: {response.text}\n") 
